# 03 Pilot run of the 3D U-Net (memory and time)

Short test on fold 0 of the development set only (the test set is never loaded here). It answers two questions for the compute budget of the report about 

1. How much GPU memory does training take for full volume vs cropped volume vs patches?
2. How long does one training run take?

The Dice obtained after this short pilot is not a result !!! 

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import torch  # deep learning library
from monai.data import CacheDataset, DataLoader  # MONAI = medical imaging tools built on torch
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.nets import UNet
from monai.transforms import (AsDiscrete, Compose, EnsureChannelFirstd, LoadImaged, NormalizeIntensityd,
                              RandAffined, RandCropByPosNegLabeld, RandFlipd, RandScaleIntensityd,
                              RandShiftIntensityd)
from monai.utils import set_determinism

DATA_ROOT = Path("../data/Task02_Heart")
splits = json.loads(Path("../splits/splits.json").read_text())

device = torch.device("cuda")  # computations on the GPU
torch.backends.cudnn.benchmark = True
set_determinism(seed=0)  # fixed random seed
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

NVIDIA GeForce RTX 4060 Laptop GPU 8.0 GB


In [ ]:
PATCH = (96, 96, 96)  # size of the 3D blocks given to the network

def make_model():
    # 3D U-Net with 5 levels; 2 outputs, background and LA
    return UNet(spatial_dims=3, in_channels=1, out_channels=2,
                channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), num_res_units=2)

# Dice + cross-entropy loss, robust to the small size of the LA
loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
print(f"Parameters: {sum(p.numel() for p in make_model().parameters()) / 1e6:.1f} M")

Parameters: 4.8 M


## 1. Memory test

One training step (forward + backward, mixed precision) on random data of each size. Peak memory above 7.5 GB means it does not fit on the 8 GB GPU 


In [3]:
def peak_memory_gb(shape, batch):
    # one training step on random data, returns the maximum GPU memory used
    model = make_model().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        x = torch.randn(batch, 1, *shape, device=device)
        y = torch.randint(0, 2, (batch, 1, *shape), device=device)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / 1024**3
    except torch.OutOfMemoryError:
        return float("inf")
    finally:
        del model, opt
        torch.cuda.empty_cache()

# the three ways of feeding the data to the network
configs = {
    "Full volume 320x320x128, batch 1": ((320, 320, 128), 1),
    "Cropped volume 160x160x96, batch 1": ((160, 160, 96), 1),
    "Patches 96^3, batch 4": (PATCH, 4),
}
for name, (shape, batch) in configs.items():
    print(f"{name:38s} peak memory = {peak_memory_gb(shape, batch):.2f} GB")

Full volume 320x320x128, batch 1       peak memory = 1.71 GB
Cropped volume 160x160x96, batch 1     peak memory = 0.34 GB
Patches 96^3, batch 4                  peak memory = 0.52 GB


## 2. Data (fold 0)

In [4]:
def as_dicts(case_list):
    # one dict per volume: path of the image and of its label
    return [{"image": str(DATA_ROOT / "imagesTr" / f"{c}.nii.gz"),
             "label": str(DATA_ROOT / "labelsTr" / f"{c}.nii.gz")} for c in case_list]

val_cases = splits["folds"][0]
train_cases = [c for c in splits["dev"] if c not in val_cases]
print(f"Train ({len(train_cases)}):", train_cases)
print(f"Val   ({len(val_cases)}):", val_cases)

# loading and normalisation, for training and validation
base = [LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        NormalizeIntensityd(keys="image", nonzero=True)]

# random patches and augmentations, for training only
train_tf = Compose(base + [
    RandCropByPosNegLabeld(keys=["image", "label"], label_key="label", spatial_size=PATCH,
                           pos=1, neg=1, num_samples=2),
    RandAffined(keys=["image", "label"], prob=0.3, rotate_range=(0.26, 0.26, 0.26),
                scale_range=(0.1, 0.1, 0.1), mode=("bilinear", "nearest")),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandScaleIntensityd(keys="image", factors=0.1, prob=0.5),
    RandShiftIntensityd(keys="image", offsets=0.1, prob=0.5),
])
val_tf = Compose(base)

# CacheDataset keeps the loaded volumes in memory
train_ds = CacheDataset(as_dicts(train_cases), train_tf, progress=False)
val_ds = CacheDataset(as_dicts(val_cases), val_tf, progress=False)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=0)  # 2 volumes x 2 patches = 4 patches

Train (12): ['la_005', 'la_007', 'la_009', 'la_011', 'la_014', 'la_017', 'la_018', 'la_021', 'la_022', 'la_024', 'la_026', 'la_029']
Val   (3): ['la_003', 'la_010', 'la_023']


## 3. Timed training

In [5]:
N_PILOT_ITERS = 300

model = make_model().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scaler = torch.amp.GradScaler("cuda")  # mixed precision (float16): less memory, faster
losses, it = [], 0
torch.cuda.reset_peak_memory_stats()

model.train()
t0 = time.perf_counter()
while it < N_PILOT_ITERS:
    for batch in train_loader:
        x, y = batch["image"].to(device), batch["label"].to(device)
        # one training step: prediction, loss, gradients, weight update
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        losses.append(loss.item())
        it += 1
        if it % 50 == 0:
            print(f"iter {it:4d}  loss {np.mean(losses[-50:]):.3f}")
        if it == N_PILOT_ITERS:
            break
torch.cuda.synchronize()
train_time = time.perf_counter() - t0
sec_per_iter = train_time / N_PILOT_ITERS
print(f"\n{sec_per_iter:.2f} s per iteration, peak memory {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

iter   50  loss 1.166
iter  100  loss 0.660
iter  150  loss 0.442
iter  200  loss 0.387
iter  250  loss 0.332
iter  300  loss 0.330

0.31 s per iteration, peak memory 0.53 GB


In [6]:
# validation on full volumes with a sliding window (also timed)
dice_metric = DiceMetric(include_background=False)
to_pred, to_label = AsDiscrete(argmax=True, to_onehot=2), AsDiscrete(to_onehot=2)

model.eval()
t0 = time.perf_counter()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    for item in val_ds:
        x = item["image"].unsqueeze(0).to(device)
        # the volume is cut into overlapping patches, predicted, then put back together
        logits = sliding_window_inference(x, PATCH, sw_batch_size=4, predictor=model, overlap=0.25)
        dice_metric([to_pred(logits[0])], [to_label(item["label"].to(device))])
torch.cuda.synchronize()
val_time = time.perf_counter() - t0
print(f"Validation of {len(val_ds)} volumes: {val_time:.1f} s")
print(f"Pilot Dice after {N_PILOT_ITERS} iterations (not a result): {dice_metric.aggregate().item():.3f}")

Validation of 3 volumes: 1.2 s
Pilot Dice after 300 iterations (not a result): 0.787


## 4. Compute budget estimate

`N_ITERS_PLANNED` is the planned length of one full training run (to adjust once we see how fast the validation Dice converges).

In [7]:
N_ITERS_PLANNED = 5000
VAL_EVERY = 250
N_RUNS = 5 + 3  # 5 cross-validation folds + 3 final seeds

# time of one run = training + a validation every VAL_EVERY iterations
hours_per_run = (sec_per_iter * N_ITERS_PLANNED + val_time * N_ITERS_PLANNED / VAL_EVERY) / 3600
print(f"One run of {N_ITERS_PLANNED} iterations: {hours_per_run:.2f} h")
print(f"Total for {N_RUNS} runs: {hours_per_run * N_RUNS:.1f} h")

One run of 5000 iterations: 0.44 h
Total for 8 runs: 3.5 h
